# Day 23: The Transformer Encoder Block

> **Goal:** Build a complete Transformer Encoder block from scratch — combining multi-head attention, feed-forward networks, layer normalization, and residual connections.

---

## Architecture Overview

A single Transformer Encoder block:

```
        ┌───────────┐
Input → │ LayerNorm │ → Multi-Head Attention → + (residual) →
        └───────────┘                          ↑
                                    Input ─────┘
        ┌───────────┐
     → │ LayerNorm │ → Feed-Forward Network → + (residual) → Output
        └───────────┘                          ↑
                                   Input ──────┘
```

This is the **Pre-Norm** variant (LayerNorm before each sub-layer), which is more stable for training.

## Recap from Day 22

Yesterday we built **multi-head self-attention**. Today we add the remaining pieces:
1. **LayerNorm** — normalize activations per token
2. **Feed-Forward Network (FFN)** — two-layer MLP with GELU
3. **Residual connections** — add input to output of each sub-layer
4. **Stacking** — repeat the block L times

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import math

torch.manual_seed(42)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

---
## 1. Layer Normalization

**BatchNorm** normalizes across the batch dimension — problematic for variable-length sequences.
**LayerNorm** normalizes across the feature dimension — applied independently per token.

```
BatchNorm: normalize across batch for each feature
LayerNorm: normalize across features for each sample/token
```

In [ ]:
# Compare BatchNorm vs LayerNorm

# Simulated sequence: (batch=2, seq_len=4, d_model=8)
x = torch.randn(2, 4, 8)

# LayerNorm normalizes across the last dimension (features)
layer_norm = nn.LayerNorm(8)
ln_out = layer_norm(x)

print("Input shape:", x.shape)
print(f"\nBefore LayerNorm — token [0,0]: mean={x[0,0].mean():.4f}, std={x[0,0].std():.4f}")
print(f"After  LayerNorm — token [0,0]: mean={ln_out[0,0].mean():.4f}, std={ln_out[0,0].std():.4f}")
print(f"\nLayerNorm has {sum(p.numel() for p in layer_norm.parameters())} params (gamma + beta)")

### Pre-Norm vs Post-Norm

| Variant | Formula | Used by |
|---|---|---|
| **Post-Norm** (original) | `LayerNorm(x + Sublayer(x))` | Original Transformer (2017) |
| **Pre-Norm** (modern) | `x + Sublayer(LayerNorm(x))` | GPT-2, ViT, most modern models |

Pre-Norm is more stable for deep stacks because the residual path is "clean" — gradients flow through additions without being normalized.

---
## 2. Feed-Forward Network (FFN)

The FFN is applied independently to each token — it's a **per-token MLP**:

```
FFN(x) = Linear₂(GELU(Linear₁(x)))
```

- **Linear₁** expands from `d_model` to `d_model × expansion_factor` (typically 4×)
- **GELU** activation (smoother than ReLU)
- **Linear₂** projects back to `d_model`

In [ ]:
class FeedForward(nn.Module):
    """Position-wise Feed-Forward Network."""

    def __init__(self, d_model, expansion=4, dropout=0.1):
        super().__init__()
        hidden = d_model * expansion
        self.net = nn.Sequential(
            nn.Linear(d_model, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

# Test
ffn = FeedForward(d_model=64, expansion=4)
x = torch.randn(2, 10, 64)  # (batch, seq_len, d_model)
out = ffn(x)
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")
print(f"FFN params: {sum(p.numel() for p in ffn.parameters()):,}")

### Why GELU instead of ReLU?

Transformers use **GELU** (Gaussian Error Linear Unit) instead of ReLU. GELU is smoother — it doesn't abruptly cut to zero, which helps gradient flow. Let's compare them visually:

In [ ]:
# Why GELU instead of ReLU?
x_vis = torch.linspace(-4, 4, 200)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x_vis.numpy(), F.relu(x_vis).numpy(), label="ReLU", linewidth=2)
ax.plot(x_vis.numpy(), F.gelu(x_vis).numpy(), label="GELU", linewidth=2)
ax.set_xlabel("Input"); ax.set_ylabel("Output")
ax.set_title("GELU vs ReLU — GELU is smoother, allows small negative values")
ax.legend(); ax.grid(True, alpha=0.3)
ax.axhline(0, color='k', linewidth=0.5); ax.axvline(0, color='k', linewidth=0.5)
plt.tight_layout(); plt.show()

---
## 3. Multi-Head Attention (from Day 22)

Quick re-implementation with the combined QKV projection (more efficient):

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-Head Self-Attention with combined QKV projection."""

    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.attn_drop = nn.Dropout(dropout)
        self.proj_drop = nn.Dropout(dropout)

    def forward(self, x, return_weights=False):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.d_k).permute(2, 0, 3, 1, 4)
        Q, K, V = qkv.unbind(0)

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_drop(attn)

        out = (attn @ V).transpose(1, 2).reshape(B, N, C)
        out = self.proj_drop(self.proj(out))

        if return_weights:
            return out, attn
        return out

# Test
mha = MultiHeadAttention(d_model=64, num_heads=4)
x = torch.randn(2, 10, 64)
out, weights = mha(x, return_weights=True)
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")
print(f"Attention weights: {weights.shape}  (batch, heads, seq, seq)")

---
## 4. The Complete Transformer Encoder Block

Now we combine everything: attention + FFN + LayerNorm + residual connections.

In [ ]:
class TransformerEncoderBlock(nn.Module):
    """A single Transformer Encoder block (Pre-Norm variant)."""

    def __init__(self, d_model, num_heads, expansion=4, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn   = FeedForward(d_model, expansion, dropout)

    def forward(self, x, return_weights=False):
        # Sub-layer 1: Multi-Head Attention with Pre-Norm + Residual
        if return_weights:
            attn_out, weights = self.attn(self.norm1(x), return_weights=True)
            x = x + attn_out
        else:
            x = x + self.attn(self.norm1(x))

        # Sub-layer 2: FFN with Pre-Norm + Residual
        x = x + self.ffn(self.norm2(x))

        if return_weights:
            return x, weights
        return x


# Test a single block
block = TransformerEncoderBlock(d_model=64, num_heads=4, expansion=4, dropout=0.1)
x = torch.randn(2, 10, 64)
out = block(x)
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")
print(f"\nBlock parameters: {sum(p.numel() for p in block.parameters()):,}")

### Parameter Breakdown

Understanding where the parameters live helps build intuition. The FFN typically has ~2/3 of all parameters in a Transformer block (because of the 4× expansion):

In [ ]:
# Parameter breakdown for one Transformer Encoder block
block = TransformerEncoderBlock(d_model=128, num_heads=4)

print("Parameter breakdown (d_model=128, 4 heads, expansion=4):")
print("=" * 55)
for name, param in block.named_parameters():
    print(f"  {name:30s} → {str(list(param.shape)):15s} = {param.numel():,}")
print("=" * 55)
print(f"  TOTAL: {sum(p.numel() for p in block.parameters()):,} parameters")

---
## 5. Stacking Transformer Encoder Blocks

A Transformer Encoder is simply N identical blocks stacked:

In [ ]:
class TransformerEncoder(nn.Module):
    """Stack of Transformer Encoder blocks."""

    def __init__(self, d_model, num_heads, num_layers, expansion=4, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, expansion, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, return_all_weights=False):
        all_weights = []
        for layer in self.layers:
            if return_all_weights:
                x, weights = layer(x, return_weights=True)
                all_weights.append(weights)
            else:
                x = layer(x)
        x = self.norm(x)
        if return_all_weights:
            return x, all_weights
        return x

# Build an encoder with 6 layers
encoder = TransformerEncoder(d_model=128, num_heads=4, num_layers=6)
x = torch.randn(2, 20, 128)
out = encoder(x)

print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")
print(f"\nTotal encoder parameters: {sum(p.numel() for p in encoder.parameters()):,}")
print(f"  Per layer: ~{sum(p.numel() for p in encoder.layers[0].parameters()):,}")

---
## 6. Verifying the Residual Path

Let's visualize how the residual connections help gradient flow:

In [ ]:
# Compare signal magnitude through the encoder with and without residuals
def test_signal_propagation(num_layers=12, d_model=128, with_residual=True):
    magnitudes = []
    x = torch.randn(1, 10, d_model)
    magnitudes.append(x.norm().item())

    for i in range(num_layers):
        block = TransformerEncoderBlock(d_model, num_heads=4)
        with torch.no_grad():
            if with_residual:
                x = block(x)
            else:
                x = block.attn(block.norm1(x))
                x = block.ffn(block.norm2(x))
        magnitudes.append(x.norm().item())
    return magnitudes

mags_with = test_signal_propagation(with_residual=True)
mags_without = test_signal_propagation(with_residual=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(mags_with, 'o-', label="With residuals", linewidth=2)
ax.plot(mags_without, 's-', label="Without residuals", linewidth=2)
ax.set_xlabel("Layer"); ax.set_ylabel("Signal magnitude (L2 norm)")
ax.set_title("Residual Connections Maintain Signal Strength")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 7. Attention Patterns in the Encoder

In [ ]:
# Visualize attention from different layers
encoder_vis = TransformerEncoder(d_model=64, num_heads=4, num_layers=4)
x = torch.randn(1, 8, 64)

with torch.no_grad():
    out, all_weights = encoder_vis(x, return_all_weights=True)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for layer_idx, (ax, weights) in enumerate(zip(axes, all_weights)):
    attn = weights[0, 0].numpy()
    im = ax.imshow(attn, cmap='viridis', vmin=0, vmax=attn.max())
    ax.set_title(f"Layer {layer_idx+1}, Head 0")
    ax.set_xlabel("Key"); ax.set_ylabel("Query")
plt.colorbar(im, ax=axes, shrink=0.8, label="Attention Weight")
plt.suptitle("Attention Patterns Across Layers (Random Init)", fontsize=14)
plt.tight_layout(); plt.show()

---
## 🧪 Exercises

### Exercise 1: Post-Norm vs Pre-Norm
Implement a Post-Norm version (norm after attention/FFN instead of before). Compare gradient magnitudes in early layers.

### Exercise 2: Different expansion ratios
Try FFN expansion factors of 1×, 2×, and 4×. How does it affect parameter count and expressiveness?

### Exercise 3: Dropout ablation
Train a small encoder with dropout rates of 0.0, 0.1, 0.3, 0.5. Which works best?

### Exercise 4: Custom encoder for classification
Add a [CLS] token to the beginning of a random sequence, pass through the encoder, and use the [CLS] output for classification on a toy task.

In [ ]:
# Exercises: Your code here

---
## 📝 Key Takeaways

| Concept | Summary |
|---|---|
| **Transformer Encoder Block** | Attention + FFN, each with LayerNorm + residual |
| **LayerNorm** | Normalizes across features (per token), not across batch |
| **FFN** | Two-layer MLP with GELU, applied per token independently |
| **Pre-Norm** | LayerNorm before sublayer — more stable training |
| **Stacking** | Just repeat the same block N times |
| **Residual connections** | Allow gradient flow through deep stacks |

**Tomorrow:** From text to images — we'll learn how to split images into patches and create patch embeddings for the Vision Transformer!